# OpenEMMA Baseline Evaluation on Kaggle

This notebook runs OpenEMMA baseline evaluation optimized for Kaggle Notebooks.

## Requirements:
- Enable GPU accelerator in Kaggle settings
- Add NuScenes mini dataset to Kaggle datasets (or use Kaggle's NuScenes dataset)
- Runtime: ~30-60 minutes depending on GPU

## What this does:
1. Sets up OpenEMMA with Kaggle-specific optimizations
2. Downloads and configures dependencies
3. Runs baseline evaluation on NuScenes mini
4. Generates comprehensive metrics and visualizations

In [ ]:
# Kaggle Environment Setup and GPU Check
import os
import subprocess
import sys

print("🔍 Kaggle Environment Check:")
print(f"Python version: {sys.version}")
print(f"Working directory: {os.getcwd()}")
print(f"Available space: {subprocess.check_output(['df', '-h', '.']).decode().split('\n')[1]}")

# Check GPU availability
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"🎮 GPU: {gpu_name}")
        print(f"💾 GPU Memory: {gpu_memory:.1f}GB")
    else:
        print("❌ No GPU available - this will be very slow!")
except ImportError:
    print("⚠️ PyTorch not available yet")

# Set Kaggle-specific environment variables
os.environ.update({
    'KAGGLE_KERNEL_RUN_TYPE': 'Interactive',
    'PYTORCH_CUDA_ALLOC_CONF': 'max_split_size_mb:512',
    'TRANSFORMERS_CACHE': '/tmp/transformers_cache',
    'HF_HOME': '/tmp/huggingface_cache',
    'CUDA_LAUNCH_BLOCKING': '1'
})

print("✅ Kaggle environment configured!")

In [ ]:
# Install Dependencies Optimized for Kaggle
print("📦 Installing dependencies for Kaggle...")

# Kaggle often has some packages pre-installed, so we install only what's needed
!pip install -q --no-cache-dir \
    torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

!pip install -q --no-cache-dir \
    transformers>=4.35.0 \
    accelerate \
    bitsandbytes \
    flash-attn \
    nuscenes-devkit \
    qwen_vl_utils \
    pyquaternion \
    opencv-python \
    Pillow \
    scipy \
    matplotlib \
    seaborn \
    pandas \
    openai

print("✅ Dependencies installed!")

# Verify critical packages
import torch
import transformers
print(f"🔧 PyTorch: {torch.__version__}")
print(f"🤗 Transformers: {transformers.__version__}")
print(f"🎮 CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Download NuScenes Dataset (Kaggle-specific)
import os
import zipfile
import requests
from pathlib import Path

# Check if NuScenes is already available in Kaggle datasets
KAGGLE_DATASETS_PATH = "/kaggle/input"
NUSCENES_DIR = None

# Common Kaggle NuScenes dataset paths
kaggle_nuscenes_paths = [
    "/kaggle/input/nuscenes-mini",
    "/kaggle/input/nuscenes", 
    "/kaggle/input/nuscenes-dataset",
    "/kaggle/input/nuscenes-mini-dataset"
]

print("🔍 Searching for NuScenes dataset in Kaggle...")
for path in kaggle_nuscenes_paths:
    if os.path.exists(path):
        print(f"✅ Found NuScenes at: {path}")
        NUSCENES_DIR = path
        # List contents to verify
        contents = os.listdir(path)
        print(f"   Contents: {contents[:5]}...")  # Show first 5 items
        break

# If not found in Kaggle datasets, download manually
if NUSCENES_DIR is None:
    print("📥 NuScenes not found in Kaggle datasets, downloading...")
    NUSCENES_DIR = "/kaggle/working/nuscenes_data"
    os.makedirs(NUSCENES_DIR, exist_ok=True)
    
    # Download NuScenes mini (lighter for Kaggle)
    !wget -q https://www.nuscenes.org/data/v1.0-mini.tgz -O /tmp/nuscenes_mini.tgz
    !tar -xf /tmp/nuscenes_mini.tgz -C {NUSCENES_DIR}
    
    print(f"✅ NuScenes downloaded to: {NUSCENES_DIR}")

# Verify dataset structure
required_dirs = ['maps', 'samples', 'sweeps', 'v1.0-mini']
missing_dirs = []

for required_dir in required_dirs:
    full_path = os.path.join(NUSCENES_DIR, required_dir)
    if os.path.exists(full_path):
        print(f"✅ Found: {required_dir}")
    else:
        missing_dirs.append(required_dir)
        print(f"❌ Missing: {required_dir}")

if missing_dirs:
    print(f"⚠️ Missing directories: {missing_dirs}")
    print("💡 You may need to add NuScenes as a Kaggle dataset or check the download.")
else:
    print(f"🎯 NuScenes dataset ready at: {NUSCENES_DIR}")

In [ ]:
# Clone OpenEMMA Repository
import os
import subprocess

# Clone the repository
if not os.path.exists('/kaggle/working/OpenEMMA'):
    print("📥 Cloning OpenEMMA repository...")
    !git clone -b baseline-evalutation https://github.com/yasinshahid/OpenEMMA.git /kaggle/working/OpenEMMA
else:
    print("✅ OpenEMMA already cloned")

# Change to OpenEMMA directory
os.chdir('/kaggle/working/OpenEMMA')
print(f"📁 Working directory: {os.getcwd()}")

# Install any additional requirements
if os.path.exists('requirements.txt'):
    print("📦 Installing OpenEMMA requirements...")
    !pip install -q --no-cache-dir -r requirements.txt

print("✅ OpenEMMA setup complete!")

In [ ]:
# Kaggle GPU Optimization and Memory Management
import torch
import gc
import psutil

def optimize_for_kaggle():
    """Apply Kaggle-specific optimizations"""
    print("🔧 Applying Kaggle optimizations...")
    
    # Clear any existing GPU memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        print("   ✅ GPU cache cleared")
        
        # Set memory fraction (Kaggle GPUs are typically P100 16GB or T4 16GB)
        torch.cuda.set_per_process_memory_fraction(0.9)  # Use 90% of available GPU memory
        print("   ✅ GPU memory fraction set to 90%")
    
    # Clear system memory
    gc.collect()
    
    # Show memory stats
    ram = psutil.virtual_memory()
    print(f"   💾 RAM: {ram.used/1e9:.1f}GB used / {ram.total/1e9:.1f}GB total ({ram.percent:.1f}% used)")
    
    if torch.cuda.is_available():
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        gpu_allocated = torch.cuda.memory_allocated() / 1e9
        print(f"   🎮 GPU: {gpu_allocated:.1f}GB used / {gpu_memory:.1f}GB total")
    
    print("✅ Kaggle optimization complete!")

# Apply optimizations
optimize_for_kaggle()

# Test quantization setup
if os.path.exists('test_quantization.py'):
    print("🧪 Testing quantization setup...")
    !python test_quantization.py
else:
    print("⚠️ Quantization test not available, but should work with main.py")

In [ ]:
# Run OpenEMMA Evaluation on Kaggle
import subprocess
import sys
from datetime import datetime

print("🚀 Starting OpenEMMA Baseline Evaluation...")
print(f"⏰ Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Prepare command with Kaggle-optimized settings
cmd = [
    sys.executable, 'main.py',
    '--model-path', 'llava',  # LLaVA works best on Kaggle
    '--quantize', '4bit',     # Maximum memory efficiency
    '--colab-mode',           # Enable memory optimizations (works for Kaggle too)
    '--dataroot', NUSCENES_DIR,
    '--version', 'v1.0-mini',
    '--method', 'openemma'
]

print(f"🔧 Command: {' '.join(cmd)}")
print(f"📁 NuScenes path: {NUSCENES_DIR}")
print(f"⚙️ Settings: LLaVA + 4-bit quantization + Kaggle optimizations")

# Run the evaluation
try:
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)  # 1 hour timeout
    
    print("\n" + "="*60)
    print("📊 EVALUATION OUTPUT:")
    print("="*60)
    print(result.stdout)
    
    if result.stderr:
        print("\n" + "="*60)
        print("⚠️ WARNINGS/ERRORS:")
        print("="*60)
        print(result.stderr)
    
    if result.returncode == 0:
        print("\n✅ Evaluation completed successfully!")
    else:
        print(f"\n❌ Evaluation failed with exit code: {result.returncode}")
        
except subprocess.TimeoutExpired:
    print("\n⏰ Evaluation timed out after 1 hour")
except Exception as e:
    print(f"\n❌ Error running evaluation: {e}")

print(f"\n⏰ End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Check for results
result_dirs = ['./llava_results/', './qwen_results/', './gpt_results/']
for result_dir in result_dirs:
    if os.path.exists(result_dir):
        print(f"📁 Results found in: {result_dir}")
        # List contents
        for root, dirs, files in os.walk(result_dir):
            if files:
                print(f"   📄 {root}: {files}")

In [ ]:
# Results Analysis for Kaggle
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
import os

def analyze_kaggle_results():
    """Analyze evaluation results optimized for Kaggle display"""
    
    # Find results directories
    result_dirs = ['./llava_results/', './qwen_results/', './gpt_results/']
    found_results = []
    
    for result_dir in result_dirs:
        if os.path.exists(result_dir):
            jsonl_files = glob(os.path.join(result_dir, "**/ade_results.jsonl"), recursive=True)
            if jsonl_files:
                found_results.append((result_dir, jsonl_files[0]))
    
    if not found_results:
        print("❌ No evaluation results found!")
        print("🔍 Checking directories:")
        for result_dir in result_dirs:
            if os.path.exists(result_dir):
                print(f"   📁 {result_dir}: {os.listdir(result_dir)}")
            else:
                print(f"   ❌ {result_dir}: Not found")
        return None
    
    # Analyze the first found results
    result_dir, results_file = found_results[0]
    model_type = "llava" if "llava" in result_dir else "qwen" if "qwen" in result_dir else "gpt"
    
    print(f"📊 Analyzing results from: {results_file}")
    print(f"🤖 Model type: {model_type.upper()}")
    
    # Load results
    results = []
    with open(results_file, 'r') as f:
        for line in f:
            if line.strip():
                results.append(json.loads(line))
    
    if not results:
        print("❌ No data found in results file")
        return None
    
    df = pd.DataFrame(results)
    
    # Display summary
    print("\n" + "="*60)
    print(f"🎯 KAGGLE BASELINE EVALUATION RESULTS ({model_type.upper()})")
    print("="*60)
    
    print(f"📈 Scenes evaluated: {len(df)}")
    print(f"📋 Scene names: {', '.join(df['name'].tolist())}")
    
    print("\n📊 Average Displacement Error (ADE) Metrics:")
    print(f"   • ADE 1s: {df['ade1s'].mean():.4f} ± {df['ade1s'].std():.4f} meters")
    print(f"   • ADE 2s: {df['ade2s'].mean():.4f} ± {df['ade2s'].std():.4f} meters") 
    print(f"   • ADE 3s: {df['ade3s'].mean():.4f} ± {df['ade3s'].std():.4f} meters")
    print(f"   • Overall: {df['avgade'].mean():.4f} ± {df['avgade'].std():.4f} meters")
    
    # Create Kaggle-optimized visualizations
    plt.style.use('default')
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f'OpenEMMA Baseline Results - {model_type.upper()} Model (Kaggle)', fontsize=16, fontweight='bold')
    
    # 1. ADE by time horizon
    metrics = ['ade1s', 'ade2s', 'ade3s']
    means = [df[m].mean() for m in metrics]
    stds = [df[m].std() for m in metrics]
    
    bars = axes[0,0].bar(['1s', '2s', '3s'], means, yerr=stds, capsize=5, 
                        color=['#3498db', '#e74c3c', '#2ecc71'], alpha=0.8)
    axes[0,0].set_title('ADE by Prediction Horizon', fontweight='bold')
    axes[0,0].set_ylabel('ADE (meters)')
    axes[0,0].grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, mean in zip(bars, means):
        axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                      f'{mean:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # 2. Per-scene comparison
    scenes = df['name'].tolist()
    x_pos = np.arange(len(scenes))
    width = 0.25
    
    axes[0,1].bar(x_pos - width, df['ade1s'], width, label='1s', color='#3498db', alpha=0.8)
    axes[0,1].bar(x_pos, df['ade2s'], width, label='2s', color='#e74c3c', alpha=0.8) 
    axes[0,1].bar(x_pos + width, df['ade3s'], width, label='3s', color='#2ecc71', alpha=0.8)
    axes[0,1].set_title('ADE by Scene', fontweight='bold')
    axes[0,1].set_ylabel('ADE (meters)')
    axes[0,1].set_xticks(x_pos)
    axes[0,1].set_xticklabels([s.replace('scene-', '') for s in scenes], rotation=45)
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # 3. Overall ADE distribution
    axes[1,0].hist(df['avgade'], bins=min(8, len(df)), alpha=0.7, color='#9b59b6', edgecolor='black')
    axes[1,0].axvline(df['avgade'].mean(), color='red', linestyle='--', linewidth=2,
                     label=f'Mean: {df["avgade"].mean():.3f}m')
    axes[1,0].set_title('Distribution of Average ADE', fontweight='bold')
    axes[1,0].set_xlabel('Average ADE (meters)')
    axes[1,0].set_ylabel('Frequency')
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)
    
    # 4. Summary statistics table
    axes[1,1].axis('off')
    summary_data = [
        ['Metric', 'Mean', 'Std Dev', 'Min', 'Max'],
        ['ADE 1s', f'{df["ade1s"].mean():.4f}', f'{df["ade1s"].std():.4f}', 
         f'{df["ade1s"].min():.4f}', f'{df["ade1s"].max():.4f}'],
        ['ADE 2s', f'{df["ade2s"].mean():.4f}', f'{df["ade2s"].std():.4f}', 
         f'{df["ade2s"].min():.4f}', f'{df["ade2s"].max():.4f}'],
        ['ADE 3s', f'{df["ade3s"].mean():.4f}', f'{df["ade3s"].std():.4f}', 
         f'{df["ade3s"].min():.4f}', f'{df["ade3s"].max():.4f}'],
        ['Overall', f'{df["avgade"].mean():.4f}', f'{df["avgade"].std():.4f}', 
         f'{df["avgade"].min():.4f}', f'{df["avgade"].max():.4f}']
    ]
    
    table = axes[1,1].table(cellText=summary_data, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    
    # Style the header row
    for i in range(len(summary_data[0])):
        table[(0, i)].set_facecolor('#34495e')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    axes[1,1].set_title('Summary Statistics', fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.show()
    
    # Save results for Kaggle output
    baseline_summary = {
        'platform': 'kaggle',
        'model': model_type,
        'method': 'openemma',
        'quantization': '4bit',
        'scenes_evaluated': len(df),
        'scene_names': df['name'].tolist(),
        'metrics': {
            'ade1s': {'mean': df['ade1s'].mean(), 'std': df['ade1s'].std()},
            'ade2s': {'mean': df['ade2s'].mean(), 'std': df['ade2s'].std()},
            'ade3s': {'mean': df['ade3s'].mean(), 'std': df['ade3s'].std()},
            'avgade': {'mean': df['avgade'].mean(), 'std': df['avgade'].std()}
        }
    }
    
    # Save to Kaggle output
    with open('/kaggle/working/baseline_results_kaggle.json', 'w') as f:
        json.dump(baseline_summary, f, indent=2)
    
    print(f"\n💾 Results saved to: /kaggle/working/baseline_results_kaggle.json")
    print("📋 Download this file for future comparison with optimized versions!")
    
    return df

# Run analysis
print("📊 Starting results analysis...")
results_df = analyze_kaggle_results()

if results_df is not None:
    print("\n🎉 Baseline evaluation complete!")
    print("✅ You now have baseline metrics to compare against future optimizations.")
else:
    print("\n❌ Could not analyze results. Check if evaluation completed successfully.")

## 🎯 Next Steps

If everything ran successfully, you now have:

1. **✅ Baseline metrics** for OpenEMMA on NuScenes mini dataset
2. **📊 Visualizations** showing performance across different time horizons
3. **💾 Saved results** in `/kaggle/working/baseline_results_kaggle.json`

### For your thesis research:
- Use these baseline metrics to measure improvements from your optimizations
- The JSON file contains all metrics for easy comparison
- Consider experimenting with different models (Qwen, GPT) by changing `--model-path`

### If you encountered issues:
1. Check that GPU is enabled in Kaggle settings
2. Verify NuScenes dataset is properly loaded
3. Try reducing quantization to 8-bit if 4-bit fails
4. Check the evaluation output above for specific error messages

### Performance tips:
- This notebook is optimized for Kaggle's GPU instances
- 4-bit quantization provides maximum memory efficiency
- Results should be comparable to Colab runs

**Happy researching! 🚀**